# NHL Trap Games

In [1]:
import sys
import os
import polars as pl
from tqdm import tqdm

base_dir = '../'
sys.path.insert(1, base_dir)

import src.config as cfg
import src.utils as utils
import src.sportsipy_utils as sportsipy_utils

## Pull Schedule for Each NHL Team

In [2]:
# Warning: NHL Schedules not loading - https://github.com/davidjkrause/sportsipy/issues/14
# sportsipy_utils.pull_nhl_schedule('TOR', 2024)

In [47]:
# https://www.naturalstattrick.com/games.php?fromseason=20242025&thruseason=20242025&stype=2&sit=5v5&loc=B&team=All&rate=n
schedule_folder = f'{base_dir}data/natural_stat_trick'

df_games = []
for filename in tqdm(os.listdir(schedule_folder)):
    if 'games_' in filename:
        schedule_filepath = os.path.join(schedule_folder, filename)
        df_games_filepath = pl.scan_csv(schedule_filepath, null_values=["-"]).collect()
        if not df_games_filepath.is_empty():
            df_games += [df_games_filepath]
    
df_games = (pl.concat(df_games)
            .unique(subset=['Game', 'Team'], keep='first')
            .select(['Game', 'Team', 'GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%' ,'PDO'])
           )
utils.logger.info(f"Loaded {df_games.shape[0]/2} unique games from natural stattrick") 
df_games.head(5)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 53.77it/s]
2025-09-13 11:29:04,508 [3133656323.py:16] [INFO] Loaded 21528.0 unique games from natural stattrick


Game,Team,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64
""" 2024-10-22 - Wild 5, Panthers…","""Minnesota Wild""",4,1,80.0,1.89,2.27,45.4,19.05,96.0,1.15
""" 2013-01-20 - Flyers 2, Sabres…","""Buffalo Sabres""",0,1,0.0,1.32,1.45,47.55,0.0,95.24,0.952
""" 2025-03-25 - Rangers 1, Kings…","""New York Rangers""",1,0,100.0,2.22,2.89,43.5,5.0,100.0,1.05
""" 2021-11-07 - Blues 1, Ducks 4""","""Anaheim Ducks""",2,1,66.67,1.87,1.66,52.94,9.09,96.15,1.052
""" 2010-11-26 - Senators 1, Peng…","""Ottawa Senators""",1,0,100.0,2.27,2.13,51.55,2.94,100.0,1.029


In [48]:
# Warning: for 1000 pairs of (game, team), there were no extracted GF/GA numbers 
df_missing = df_games.filter(pl.col("GF%").is_null())
utils.logger.info(f"Filling missing GF% data for {df_missing.shape[0]} rows")

df_missing = (
    df_missing.with_columns([
        pl.col("Game").str.split(" - ").list.get(1).str.strip_chars().str.split(",").alias("matchup"),
        pl.col("Team").str.split(" ").list.get(-1).str.strip_chars().alias("matchup_team_name")
    ])
    .with_columns(
        # Match the team's name with the game string to extract GF
        pl.struct(["matchup", "matchup_team_name"]).map_elements(
            lambda row: next((m for m in row["matchup"] if row["matchup_team_name"] in m), None),
            return_dtype=pl.Utf8,
        ).str.strip_chars().alias("matchup_team_score")
    )
    .with_columns(
        pl.col("matchup_team_score").str.extract(r"(\d+)$").cast(pl.Int64).alias("GF")
    )
    .with_columns(
        # Match the game string not containing the team's name to extract GF
        pl.struct(["matchup", "matchup_team_name"]).map_elements(
            lambda row: next((m for m in row["matchup"] if row["matchup_team_name"] not in m), None),
            return_dtype=pl.Utf8,
        ).str.strip_chars().alias("matchup_opp_score")
    )
    .with_columns(
        pl.col("matchup_opp_score").str.extract(r"(\d+)$").cast(pl.Int64).alias("GA")
    )
    .with_columns(
        (pl.col("GF") / (pl.col("GF") + pl.col("GA"))).fill_nan(None).alias("GF%")
    )
)

utils.logger.info(f"Successfully filled missing GF% data for {df_missing.filter(pl.col('GF%').is_not_null()).shape[0]} / {df_missing.shape[0]} rows")
df_missing.head(5)

2025-09-13 11:29:05,212 [3515188891.py:3] [INFO] Filling missing GF% data for 1050 rows
2025-09-13 11:29:05,224 [3515188891.py:35] [INFO] Successfully filled missing GF% data for 1048 / 1050 rows


Game,Team,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,matchup,matchup_team_name,matchup_team_score,matchup_opp_score
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,list[str],str,str,str
""" 2016-11-05 - Wild 0, Avalanch…","""Colorado Avalanche""",1,0,1.0,1.13,1.13,50.18,0.0,100.0,1.0,"[""Wild 0"", "" Avalanche 1""]","""Avalanche""","""Avalanche 1""","""Wild 0"""
""" 2016-02-12 - Flames 1, Coyote…","""Arizona Coyotes""",4,1,0.8,1.54,0.67,69.68,0.0,100.0,1.0,"[""Flames 1"", "" Coyotes 4""]","""Coyotes""","""Coyotes 4""","""Flames 1"""
""" 2012-03-15 - Avalanche 0, Dev…","""New Jersey Devils""",1,0,1.0,2.71,1.97,57.99,0.0,100.0,1.0,"[""Avalanche 0"", "" Devils 1""]","""Devils""","""Devils 1""","""Avalanche 0"""
""" 2024-03-30 - Hurricanes 3, Ca…","""Montreal Canadiens""",0,3,0.0,1.7,2.09,44.84,0.0,100.0,1.0,"[""Hurricanes 3"", "" Canadiens 0""]","""Canadiens""","""Canadiens 0""","""Hurricanes 3"""
""" 2018-01-07 - Panthers 2, Blue…","""Columbus Blue Jackets""",3,2,0.6,3.45,2.03,62.89,0.0,100.0,1.0,"[""Panthers 2"", "" Blue Jackets 3""]","""Jackets""","""Blue Jackets 3""","""Panthers 2"""


In [49]:
df_games_processed = (pl.concat([df_missing.select(df_games.columns), df_games])
                      .unique(subset=['Game', 'Team'], keep='first')
                      .sort("Game", descending=True)
                     )  
                     
df_games_processed.head(5)

Game,Team,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64
""" 2025-04-17 - Red Wings 3, Map…","""Toronto Maple Leafs""",2,2,50.0,1.93,2.25,46.16,12.5,93.1,1.056
""" 2025-04-17 - Red Wings 3, Map…","""Detroit Red Wings""",2,2,50.0,2.25,1.93,53.84,6.9,87.5,0.944
""" 2025-04-17 - Lightning 0, Ran…","""Tampa Bay Lightning""",0,3,0.0,2.17,1.89,53.49,0.0,85.0,0.85
""" 2025-04-17 - Lightning 0, Ran…","""New York Rangers""",3,0,100.0,1.89,2.17,46.51,15.0,100.0,1.15
""" 2025-04-17 - Islanders 1, Blu…","""Columbus Blue Jackets""",6,1,85.71,2.9,2.92,49.86,23.08,97.14,1.202
